In [1]:
import numpy as np
from aeon.classification.interval_based import RSTSF
from sklearn.metrics import accuracy_score
from tscglue.models import LokyStackerV7, LokyStackerV7SoftFilterRidge, TSCGlue, LokyStackerV8Base, LokyStackerV8AutoBestStacking, LokyStackerV8AutoBestBase
from tscglue.old_models import StackerV4, LokyStackerV5, LokyStackerV5SoftRF, LokyStackerV6
from tscglue import utils, data_loader
import polars as pl

In [2]:
dataset = "Worms"
# dataset = 'Car'
# dataset = 'HandOutlines'
# dataset = 'UWaveGestureLibraryY'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(181, 1, 900) (181,) (77, 1, 900) (77,)


In [3]:
#m = TSCGlue(random_state=492, n_repetitions=1, n_jobs=16, keep_features=True, verbose=2)
#m.fit(X_train, y_train)
#y_pred = m.predict(X_test)

In [4]:
m = LokyStackerV8Base(random_state=492, n_repetitions=2, n_jobs=16, keep_features=True, verbose=2)
m.fit(X_train, y_train)

[0.00s] Starting executor with 16 workers, run_dir=./tscglue_runs/653c6d68613c456c
[0.00s] Saved X and y to disk in 0.00s
[0.35s] Computed QUANT features (181, 7987) (11.03 MB) in 0.3520s
[0.39s] Starting repetition 0
[2.27s] Computed MultiRocket features (181, 49728) (68.67 MB) in 1.8805s
[4.28s] Computed Hydra features (181, 7168) (9.90 MB) in 2.0069s
[12.11s] Computed RDST features (181, 30000) (41.43 MB) in 7.8285s
[12.11s] Starting training with 16 workers for 40 models
[12.93s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.8131s for f-3/r-0 (3.06 MB)
[12.94s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.8201s for f-0/r-0 (3.06 MB)
[12.94s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.8228s for f-6/r-0 (3.06 MB)
[12.94s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.8238s for f-9/r-0 (3.06 MB)
[13.00s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.8836s for f-5/r-0 (3.06 MB)
[13.01s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.8924s for f-1/r-0 (3.06 MB)
[13.

,random_state,492
,n_repetitions,2
,k_folds,10
,n_jobs,16
,keep_features,True
,verbose,2


In [5]:
preds = m.predict_per_model(X_test)
tests_accs = []
for model_name, model_preds in preds.items():
    acc = accuracy_score(y_test, model_preds)
    tests_accs.append({"model": model_name, "test_accuracy": acc})
test_df = pl.DataFrame(tests_accs)

[0.0000s] Starting prediction
Starting executor with 16 workers, run_dir=./tscglue_runs/653c6d68613c456c
[6.2444s] Computed features for prediction
[6.2915s] Feature arrays saved to mmap files
[6.2916s] Starting prediction with 16 workers for 80 first-level models
[13.3535s] Predicted multirockethydra-bestk-p-ridgecv_r1 in 0.0951s
[13.3952s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.1023s
[13.4534s] Predicted multirockethydra-bestk-p-ridgecv_r1 in 0.0956s
[13.4985s] Predicted multirockethydra-bestk-p-ridgecv_r1 in 0.0984s
[13.5521s] Predicted multirockethydra-bestk-p-ridgecv_r1 in 0.1067s
[13.5546s] Predicted multirockethydra-bestk-p-ridgecv_r1 in 0.0962s
[13.5894s] Predicted quant-etc_r0 in 0.0206s
[13.5923s] Predicted quant-etc_r0 in 0.0246s
[13.6008s] Predicted multirockethydra-bestk-p-ridgecv_r1 in 0.0980s
[13.6224s] Predicted quant-etc_r0 in 0.0203s
[13.6321s] Predicted quant-etc_r0 in 0.0230s
[13.6366s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.1239s
[13.6427s] 

In [9]:
preds.keys()

dict_keys(['multirockethydra-bestk-p-ridgecv_r0', 'multirockethydra-bestk-p-ridgecv_r1', 'quant-etc_r0', 'quant-etc_r1', 'rdst-p-ridgecv_r0', 'rdst-p-ridgecv_r1', 'rstsf_r0', 'rstsf_r1', 'probability-ridgecv', 'probability-et', 'probability-rf'])

In [6]:
pl.DataFrame(m.summary())

model,level,oof_accuracy
str,i64,f64
"""multirockethydra-bestk-p-ridge…",0,0.756906
"""rdst-p-ridgecv_r0""",0,0.729282
"""quant-etc_r0""",0,0.718232
"""rstsf_r0""",0,0.723757
"""multirockethydra-bestk-p-ridge…",0,0.773481
…,…,…
"""probability-ridgecv_r1""",1,0.751381
"""probability-et_r0""",1,0.756906
"""probability-et_r1""",1,0.779006


In [7]:
pl.DataFrame(m.summary()).join(test_df, on="model").sort("test_accuracy", descending=True)

model,level,oof_accuracy,test_accuracy
str,i64,f64,f64
"""multirockethydra-bestk-p-ridge…",0,0.756906,0.805195
"""rstsf_r0""",0,0.723757,0.792208
"""rstsf_r1""",0,0.723757,0.792208
"""multirockethydra-bestk-p-ridge…",0,0.773481,0.779221
"""quant-etc_r1""",0,0.707182,0.779221
"""quant-etc_r0""",0,0.718232,0.766234
"""rdst-p-ridgecv_r1""",0,0.723757,0.714286
"""rdst-p-ridgecv_r0""",0,0.729282,0.688312


In [8]:
sfsf=efwef

NameError: name 'efwef' is not defined

In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

In [ ]:
F = m.get_features()
F.estimated_size('gb')

In [ ]:
P = m.get_oof_predictions()
P.estimated_size('gb')

In [ ]:
m = LokyStackerV6(random_state=492, n_repetitions=1, n_jobs=16)
m.fit(X_train, y_train)

In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

In [ ]:
m = LokyStackerV5(random_state=492, n_repetitions=1, n_jobs=16)
m.fit(X_train, y_train)

In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")